### Imports e Pré-Processamento de Dados

In [ ]:
# Imports dos dados
import numpy as np
import pandas as pd
from datetime import datetime
from feature_eng import split_data, input_h2h_features

In [ ]:
# Imports do Modelo

In [ ]:
df_raw = pd.read_csv('../cleaned_data/df_odds.csv', parse_dates=['date'], low_memory=False)

cols_to_save = [
    "event_id","league","date",
    "away_team","away_player","away_score",
    "home_team","home_player","home_score",
    "total_score"
    ]

df_raw = df_raw.sort_values("date").reset_index(drop=True)

df_raw = df_raw[df_raw['league'] == 22614]

In [ ]:
# Filtrar por data
df_raw = df_raw[df_raw['date'] >= datetime(2025,6,1)]

In [ ]:
train_raw, test_raw = split_data(
    df=df_raw,
    split_date=datetime(2025,8,1),
    date_column='date'
)

train_with_avg, test_with_avg, scaler = input_h2h_features(
    train_raw,
    test_raw,
    return_scaler=True,
    drop_h2h=10
)

### Definições Iniciais para o modelo

In [ ]:
features = [
    "avg_h2h",
    "median_h2h",
    "h2h_count",
    "avg_home_player",
    "avg_away_player",
    "lag_1", "lag_2", "lag_3", 
    # Dados Categóricos:
    "away_team","away_player",
    "home_team","home_player",
    # "lag_4", "lag_5",
    # "lag_6", "lag_7", "lag_8", "lag_9", "lag_10",
    # "lag_11", "lag_12", "lag_13", "lag_14", "lag_15",
    # "lag_16", "lag_17", "lag_18", "lag_19", "lag_20"
]

In [ ]:
num_features = [
    "avg_h2h","median_h2h","h2h_count",
    "avg_home_player","avg_away_player",
    "lag_1","lag_2","lag_3",
]

cat_features = ["away_team","away_player","home_team","home_player"]

In [ ]:
# Sanitiza strings (sem cast p/ int!)
for c in cat_features:
    train_with_avg[c] = train_with_avg[c].astype(str).fillna("[OOV]")
    test_with_avg[c]  = test_with_avg[c].astype(str).fillna("[OOV]")

X_num_train = train_with_avg[num_features].astype('float32')
X_num_test  = test_with_avg[num_features].astype('float32')

y_train = train_with_avg['total_score'].astype('int32').values
y_test  = test_with_avg['total_score'].astype('int32').values


In [ ]:
X_train = train_with_avg[features]
y_train = train_with_avg['total_score']

X_test = test_with_avg[features]
y_test = test_with_avg['total_score']

### Treinamento do Modelo

In [ ]:
X = X_train.values
y = y_train.values

In [41]:
# =========================
# 1) Imports e dados
# =========================
import numpy as np
import pandas as pd
from datetime import datetime
from feature_eng import split_data, input_h2h_features

# Carregar
df_raw = pd.read_csv('../cleaned_data/df_odds.csv', parse_dates=['date'], low_memory=False)
df_raw = df_raw.sort_values("date").reset_index(drop=True)
df_raw = df_raw[df_raw['league'] == 22614]
df_raw = df_raw[df_raw['date'] >= datetime(2025, 6, 1)]

train_raw, test_raw = split_data(
    df=df_raw,
    split_date=datetime(2025, 8, 1),
    date_column='date'
)

train_with_avg, test_with_avg, scaler = input_h2h_features(
    train_raw, test_raw, return_scaler=True, drop_h2h=10
)

# =========================
# 2) Features
# =========================
num_features = [
    "avg_h2h","median_h2h","h2h_count",
    "avg_home_player","avg_away_player",
    "lag_1","lag_2","lag_3",
]
cat_features = ["away_team","away_player","home_team","home_player"]

# Sanitiza strings das categóricas
for c in cat_features:
    train_with_avg[c] = train_with_avg[c].astype(str).fillna("[OOV]")
    test_with_avg[c]  = test_with_avg[c].astype(str).fillna("[OOV]")

# Numéricas -> float32 e trata NaN
X_num_train_df = train_with_avg[num_features].copy()
X_num_test_df  = test_with_avg[num_features].copy()
X_num_train_df = X_num_train_df.fillna(X_num_train_df.median(numeric_only=True))
X_num_test_df  = X_num_test_df.fillna(X_num_train_df.median(numeric_only=True))

X_num_train = X_num_train_df.values.astype('float32')
X_num_test  = X_num_test_df.values.astype('float32')

# Alvo (inteiro, não-negativo)
y_train = train_with_avg['total_score'].astype('int32').values
y_test  = test_with_avg['total_score'].astype('int32').values

print("y_train stats:", y_train.min(), y_train.max(), y_train.mean())

# =========================
# 3) Modelo Keras + TFP (NegBin) com Normalization e teto p/ μ
# =========================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
import tensorflow_probability as tfp
tfd = tfp.distributions

# Normalização numérica
norm = layers.Normalization(axis=-1)
norm.adapt(X_num_train)

# Entradas
inp_num = tf.keras.Input(shape=(len(num_features),), name='num')
x_num = norm(inp_num)

# Lookups para categóricas (string)
lookups = {}
for col in cat_features:
    lk = layers.StringLookup(oov_token='[OOV]')
    lk.adapt(train_with_avg[col].values)  # vocabulário do treino
    lookups[col] = lk

# Inputs categóricos + embeddings
emb_dims = 16  # pode ajustar para 8/16
embs, inp_cats = [], []
for col in cat_features:
    inp = tf.keras.Input(shape=(1,), dtype=tf.string, name=col)
    inp_cats.append(inp)
    idx = lookups[col](inp)                              # string -> índice
    vocab_size = lookups[col].vocabulary_size()
    emb = layers.Embedding(input_dim=vocab_size, output_dim=emb_dims, name=f"emb_{col}")(idx)
    emb = layers.Flatten()(emb)
    embs.append(emb)

# Bloco denso
h = layers.Concatenate()([x_num] + embs)
h = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(h)
h = layers.Dropout(0.2)(h)
h = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(h)

# Saída "bruta" (2 neurônios)
raw_out = layers.Dense(2, name="raw_out")(h)

# μ com teto e α>0
mu_cap = float(np.percentile(y_train, 99.5) + 3.0)  # teto plausível
mu_cap_tf = tf.constant(mu_cap, dtype=tf.float32)
print("mu_cap (teto de μ):", mu_cap)

def map_params(raw):
    z_mu = raw[:, 0:1]
    z_al = raw[:, 1:2]
    mean  = tf.clip_by_value(tf.sigmoid(z_mu) * mu_cap_tf, 1e-6, mu_cap_tf)  # (0, mu_cap]
    alpha = tf.nn.softplus(z_al) + 1e-6                                      # > 0
    return tf.concat([mean, alpha], axis=1)

params = layers.Lambda(map_params, name="params")(raw_out)

# Perda NegBin estável (Var = μ + α μ²)
@tf.function
def nbinom_loss(y_true, params):
    y_true = tf.cast(tf.reshape(y_true, (-1,)), tf.float32)
    mean   = params[:, 0]
    alpha  = params[:, 1]
    r = 1.0 / alpha
    p = tf.clip_by_value(r / (r + mean), 1e-6, 1.0 - 1e-6)
    dist = tfd.NegativeBinomial(total_count=r, probs=p)
    return -tf.reduce_mean(dist.log_prob(y_true))

model = models.Model(inputs=[inp_num] + inp_cats, outputs=params, name="negbin_mu_capped")

# =========================
# 4) Inicialização de viés (âncora na média real)
# =========================
# Chamamos o modelo 1x com tensores válidos para construir pesos
def to_str_tensor(series):
    return tf.constant(series.astype(str).values.reshape(-1, 1), dtype=tf.string)

dummy = {
    "num": np.zeros((1, len(num_features)), dtype=np.float32)
}
for c in cat_features:
    dummy[c] = tf.constant([["[OOV]"]], dtype=tf.string)

_ = model(dummy)  # build

# Define viés para μ e α
mu0 = float(y_train.mean())
mu0 = min(mu0, mu_cap * 0.9)  # evita saturar no teto
b_mu = np.log(mu0 / (mu_cap - mu0 + 1e-6))  # logit(mu0/mu_cap)
b_al = np.log(np.expm1(0.5))                # α inicial ~ 0.5

raw_layer = model.get_layer("raw_out")
W, b = raw_layer.get_weights()
b = np.array([b_mu, b_al], dtype=np.float32)
raw_layer.set_weights([W, b])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=nbinom_loss)

# =========================
# 5) Monta dicts de entrada (sem dtype=object!!)
# =========================
def make_x_dict(df_num_vals, df_cat_frame):
    x = {"num": df_num_vals}
    for c in cat_features:
        x[c] = tf.constant(df_cat_frame[c].astype(str).values.reshape(-1, 1), dtype=tf.string)
    return x

x_train = make_x_dict(X_num_train, train_with_avg)
x_val   = make_x_dict(X_num_test,  test_with_avg)

# Callback de diagnóstico
class MeanPredCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
    def on_epoch_end(self, epoch, logs=None):
        params_val = self.model.predict(self.x_val, verbose=0)
        mean_val = params_val[:, 0].mean()
        print(f"[Diag] epoch {epoch:02d} | y_val_mean={self.y_val.mean():.3f} | mean_pred_val={mean_val:.3f}")

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    MeanPredCallback(x_val=x_val, y_val=y_test),
]

# =========================
# 6) Treino
# =========================
history = model.fit(
    x=x_train,
    y=y_train,
    validation_data=(x_val, y_test),
    epochs=60,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# =========================
# 7) Previsões (μ e α) + checagem
# =========================
params_pred = model.predict(x_val, verbose=0)
mean_pred  = params_pred[:, 0]
alpha_pred = params_pred[:, 1]

print("Sanity check final:")
print("  y_test mean:", float(y_test.mean()))
print("  mean_pred mean:", float(mean_pred.mean()))
print("  μ range:", (float(mean_pred.min()), float(mean_pred.max())))
print("  α mediana:", float(np.median(alpha_pred)))

# =========================
# 8) Exemplo PMF 0..10
# =========================
r0 = 1.0 / alpha_pred[0]
p0 = r0 / (r0 + mean_pred[0])
dist0 = tfd.NegativeBinomial(total_count=r0, probs=p0)
print("μ[0]:", float(mean_pred[0]), "| α[0]:", float(alpha_pred[0]))
print("PMF[0..10]:", dist0.prob(tf.range(0, 11)).numpy())


y_train stats: 0 18 5.301478320034638
mu_cap (teto de μ): 15.0


ValueError: In a nested call() argument, you cannot mix tensors and non-tensors. Received invalid mixed argument: inputs={'num': array([[0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32), 'away_team': <tf.Tensor: shape=(1, 1), dtype=string, numpy=array([[b'[OOV]']], dtype=object)>, 'away_player': <tf.Tensor: shape=(1, 1), dtype=string, numpy=array([[b'[OOV]']], dtype=object)>, 'home_team': <tf.Tensor: shape=(1, 1), dtype=string, numpy=array([[b'[OOV]']], dtype=object)>, 'home_player': <tf.Tensor: shape=(1, 1), dtype=string, numpy=array([[b'[OOV]']], dtype=object)>}

### Avaliação em Backtesting do Modelo

In [ ]:
from backtest import Backtester

In [ ]:
from backtest import Backtester
tester = Backtester(df=test_with_avg, y_pred=mean_pred)

In [ ]:
tester.make_backtest()

### Salvar objetos do Modelo

In [ ]:
# Salvar Modelo Final
# X_full = np.concatenate([X, X_test], axis=0)
# y_full = np.concatenate([y, y_test], axis=0)

# final_model = xgb.XGBRegressor(**study.best_params)
# final_model.fit(X_full, y_full)
# final_model.save_model("model_22614.json")


# joblib.dump(scaler, "scaler_22614.pkl")

# print("✅ Modelo e scaler salvos com sucesso!")
